In [19]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import scipy.constants as pc
import astropy.units as u
from astropy.io import fits
from astropy.coordinates import SkyCoord
from astropy.table import Table
from astropy.table import vstack
from astropy.cosmology import FlatLambdaCDM
from scipy.stats import ttest_rel
from scipy.special import erf
from scipy.special import erfc


def red_chi_squared(y_values, y_uncertainties, model_data, dof):

  chi_squared = np.sum(((model_data - y_values) / y_uncertainties)**2)
  
  red_chi_sq = chi_squared / (len(y_values) - dof)

  return red_chi_sq


In [20]:
def bad_way_to_calculate_total_ha_lum(bh_mass, bh_mass_uncertainty, fwhm, fwhm_upper_err, fwhm_lower_err):
    """
    Calculate the total H-alpha luminosity from fwhm and black hole mass, using the empirical relation from Reines & Volonteri.
    See page 2, eq 1 of Reines & Volonteri (2015) for the relation between H-alpha luminosity, black hole mass and Ha broad-line fwhm.

    Needs bh_mass and errs are not in log10.

    This isnt the best.
    """
    e = 1.075
    #See lab book
    print("bh mass: ", bh_mass)
    log10_L_Ha = ( np.log10(bh_mass) - np.log10(e) - 6.57 - 2.06 * np.log10(fwhm / 1000) ) / 0.47
    print(f"Calculated log10_L_Ha: {log10_L_Ha}")

    L_Ha = ( 10**log10_L_Ha ) * 1e42
    print(f"Calculated L_Ha: {L_Ha} erg/s")
    # Converts from dimensionless10^42 to erg/s

    # Errors:
    err_log10_fwhm_upper = np.log10(fwhm + fwhm_upper_err) - np.log10(fwhm)
    err_log10_fwhm_lower = np.log10(fwhm) - np.log10(fwhm - fwhm_lower_err)

    err_log10_L_Ha_upper = (1/0.47) * np.sqrt( (np.log10(bh_mass + bh_mass_uncertainty) - np.log10(bh_mass))**2 + (err_log10_fwhm_lower * 2.06)**2 )
    err_log10_L_Ha_lower = (1/0.47) * np.sqrt( (np.log10(bh_mass) - np.log10(bh_mass - bh_mass_uncertainty))**2 + (err_log10_fwhm_upper * 2.06)**2 )

    L_Ha_upper_err = L_Ha * ( 10**err_log10_L_Ha_upper - 1 )
    print(f"Calculated L_Ha_upper_err: {L_Ha_upper_err} erg/s")
    L_Ha_lower_err = L_Ha * ( 1 - 10**(-err_log10_L_Ha_lower) )
    print(f"Calculated L_Ha_lower_err: {L_Ha_lower_err} erg/s")

    return L_Ha, L_Ha_upper_err, L_Ha_lower_err

In [21]:
def sample_split_normal(mu, sigma_minus, sigma_plus, n):
    """
    Draw samples from a split-normal (two-piece Gaussian).

    Parameters
    ----------
    mu : float
        Median/best-fit value.
    sigma_minus : float
        16th percentile uncertainty.
    sigma_plus : float
        84th percentile uncertainty.
    n : int
        Number of samples.
    """

    # Probability of drawing from each side
    p_left = sigma_minus / (sigma_minus + sigma_plus)

    u = np.random.rand(n)

    samples = np.empty(n)

    left = u < p_left

    samples[left] = (
        mu
        - np.abs(np.random.normal(
            0,
            sigma_minus,
            np.sum(left)
        ))
    )

    samples[~left] = (
        mu
        + np.abs(np.random.normal(
            0,
            sigma_plus,
            np.sum(~left)
        ))
    )

    return samples

In [22]:
def sample_log_normal_from_percentiles(median, lower_err, upper_err, n_samples):
    """
    Log-normal sampler using 16th/50th/84th percentiles.

    Parameters
    ----------
    median : float
        FWHM median value
    lower_err : float
        median - 16th percentile
    upper_err : float
        84th percentile - median
    """

    # convert to percentiles
    f16 = median - lower_err
    f84 = median + upper_err

    # avoid invalid values
    f16 = max(f16, 1e-10)
    f84 = max(f84, f16 * 1.01)

    mu = np.log10(median)
    sigma = (np.log10(f84) - np.log10(f16)) / 2

    samples_log = np.random.normal(mu, sigma, n_samples)

    return 10**samples_log

In [23]:
def fwhm_equation(samples):
    """
    Calculate FWHM from sigma and centre of a Gaussian.

    Parameters
    ----------
    samples : array-like
        Array of shape (n_samples, 2) where each row contains (sigma, centre).

    Returns
    -------
    fwhm : float
        Full width at half maximum in km/s.
    """
    sigma = samples[:, 0]
    centre = samples[:, 1]
    fwhm = 2 * np.sqrt(2 * np.log(2)) * sigma * 1e5 / centre

    return fwhm

In [24]:
def Reines_Volonteri_equation(samples):
    """
    samples shape: (n_samples, 2)
    columns = [L_BOL, FWHM]
    """

    L_bol = samples[:, 0] / 1e42 # 10^42 erg/s
    fwhm = samples[:, 1] / 1e3 # 10^3 km/s

    e = 1.075

    log10_M_BH = np.log10(e) + 6.57 + 0.47 * np.log10(L_bol) + 2.06 * np.log10(fwhm / 1000)

    return log10_M_BH

In [25]:
def uncertainty_propagation(parameters, equation, n_samples=2000, batch_size=500):
    """
    Bootstrap-based uncertainty propagation.

    Parameters
    ----------
    parameters : list of tuples
        Each element is (Q16, Q50, Q84) for a parameter.
    equation : callable
        Function that accepts arrays of sampled parameters.
    n_samples : int
        Number of Monte Carlo samples.
    batch_size : int
        Number of samples to evaluate at once.

    Returns
    -------
    med : float
        Median of propagated quantity.
    upper_err : float
        +1 sigma (84th - median)
    lower_err : float
        -1 sigma (median - 16th)
    """

    samples = []
    # Stores Monte Carlo samples for each parameter

    # Bootstrap each parameter from its empirical percentiles
    for (q16, q50, q84) in parameters:

        # construct empirical distribution (no shape assumptions)
        x = np.array([q16, q50, q84]) # what the percentile values are
        p = np.array([0.16, 0.50, 0.84]) # defines the values as the percentiles we know them to be

        # bootstrap sampling via inverse CDF interpolation
        u = np.random.rand(n_samples) # Random percentiles of 0s and 1s
        s = np.interp(u, p, x) # Takes random percentile and interpolates with known quantities and returns the interpolated value

        samples.append(s)

    # shape: (n_params, n_samples)
    samples = np.array(samples) # converts list into array

    # transpose → (n_samples, n_params)
    samples = samples.T # now each row is (param1_i, param2_i,...)

    # Evaluate model in batches
    values = []

    for start in range(0, n_samples, batch_size):

        stop = min(start + batch_size, n_samples)

        batch_samples = samples[start:stop]

        batch_values = equation(batch_samples) # rows of output_i = equation(param1_i, param2_i,...)

        values.append(batch_values)

    values = np.concatenate(values)

    # Return percentiles
    med = np.percentile(values, 50)
    low = np.percentile(values, 16)
    high = np.percentile(values, 84)

    return med, high - med, med - low

In [26]:
def open_shay_table(file_path):
        # Open the FITS file
        with fits.open(file_path) as hdul:
            # Show the HDU structure
            # hdul.info()
            # Usually the table is in extension 1
            data = hdul[1].data

        # Convert to an Astropy Table
        tbl = Table(data)
        #print("Number of objects = ", len(tbl))

        # Print column names
        # print("\nColumns:")
        # print(tbl.colnames)

        return tbl

Ha_table = open_shay_table("/nvme/scratch/work/alberttg/Summer_project/Ha_mass_BIC_DJA_err_asym")
# All Shay's data from Ha line fitting
NII_table = open_shay_table("/nvme/scratch/work/alberttg/Summer_project/NII_mass_BIC_DJA_err_asym")
# All Shay's data from Ha and NII line fitting
REDSHIFT_HA = Ha_table['redshift_2']
REDSHIFT_NII = NII_table['redshift_2'] 

In [27]:
def calculate_fwhm(table):
    """
    Calculate the FWHM and its uncertainties for both the convolved model and the gaussian model. 
    Then adds these values to the table.
    """

    # For convolved model

    sigma50 = np.asarray(table["sigma50"], dtype=float)
    centre50 = np.asarray(table["centre50"], dtype=float)
    sigma84 = np.asarray(table["sigma84"], dtype=float)
    sigma16 = np.asarray(table["sigma16"], dtype=float)
    centre84 = np.asarray(table["centre84"], dtype=float)
    centre16 = np.asarray(table["centre16"], dtype=float)

    fwhm = np.empty(len(sigma50))
    fwhm_upper_err = np.empty(len(sigma50))
    fwhm_lower_err = np.empty(len(sigma50))

    for i in range(len(sigma50)):
        
        fwhm[i], fwhm_upper_err[i], fwhm_lower_err[i] = uncertainty_propagation([[sigma16[i], sigma50[i], sigma84[i]], [centre16[i], centre50[i], centre84[i]]], fwhm_equation)

    
    # For gaussian model
    gauss_sigma50 = np.asarray(table["gauss_sigma50"], dtype=float)
    gauss_centre50 = np.asarray(table["gauss_centre50"], dtype=float)
    gauss_sigma84 = np.asarray(table["gauss_sigma84"], dtype=float)
    gauss_sigma16 = np.asarray(table["gauss_sigma16"], dtype=float)
    gauss_centre84 = np.asarray(table["gauss_centre84"], dtype=float)
    gauss_centre16 = np.asarray(table["gauss_centre16"], dtype=float)

    gauss_fwhm = np.empty(len(gauss_sigma50))
    gauss_fwhm_upper_err = np.empty(len(gauss_sigma50))
    gauss_fwhm_lower_err = np.empty(len(gauss_sigma50))

    for i in range(len(gauss_sigma50)):
        gauss_fwhm[i], gauss_fwhm_upper_err[i], gauss_fwhm_lower_err[i] = uncertainty_propagation([[gauss_sigma16[i], gauss_sigma50[i], gauss_sigma84[i]], [gauss_centre16[i], gauss_centre50[i], gauss_centre84[i]]], fwhm_equation)

    table["fwhm"] = fwhm
    table["fwhm_upper_err"] = fwhm_upper_err
    table["fwhm_lower_err"] = fwhm_lower_err
    table["gauss_fwhm"] = gauss_fwhm
    table["gauss_fwhm_upper_err"] = gauss_fwhm_upper_err
    table["gauss_fwhm_lower_err"] = gauss_fwhm_lower_err

    return table

In [28]:
def gaussian(x, amplitude, sigma, mu):
    # Precompute constants
    x = x - mu
    f = amplitude * np.exp(-(x**2)/(2*sigma**2))
    return f

def gaussXExp(x, amplitude, sigma, v, mu):
    # Precompute constants
    x = x - mu
    #v = 1.0 / 10.0**decay # lambda, corresponding to an e-folding scale
    prefactor = amplitude * (np.pi * sigma**2) / v**3

    # Argument for the exponential and error functions in each term
    exparg1 = 0.5 * sigma**2 * v**(-2) - x/v
    exparg2 = 0.5 * sigma**2 * v**(-2) + x/v

    erfarg1 = (x/sigma - sigma/v)/np.sqrt(2)
    erfarg2 = -(x/sigma + sigma/v)/np.sqrt(2)
    # Compute each part of the expression
    term1 = np.exp(exparg1) * 0.5 * erfc(-erfarg1)
    term2 = np.exp(exparg2) * 0.5 * erfc(-erfarg2)
    f = prefactor * (term1 + term2)
    return f

WAVELENGTH = np.arange(6200, 6900, 0.1)
COSMO = FlatLambdaCDM(H0=70, Om0=0.3)

def line_integral(x_range, y_values, redshift, cosmo):
    """
    Integrate one or many spectra.

    Parameters
    ----------
    x_range : (Nwave,)
    y_values : (Nwave,) or (Nsamples, Nwave)
    """

    distance = cosmo.comoving_distance(redshift).value

    # Trapezoid quantities
    dlambda = np.diff(x_range)
    lambda_mid = 0.5 * (x_range[:-1] + x_range[1:])

    avg_flux = 0.5 * (y_values[..., :-1] + y_values[..., 1:])

    # Convert F_lambda
    f_lambda = avg_flux * dlambda * 3e18 / lambda_mid**2

    # SI
    f_lambda *= 1e-29

    factor = (
        4 * np.pi *
        (distance * 3.086e22 * 1e2)**2 *
        (1 + redshift)
    )

    luminosity = f_lambda * factor
    
    return np.sum(luminosity, axis=-1)


def L_BH_conv(samples, redshift):
    """
    Calculates BH Halpha bolometric luminosity in erg/s for convolved broad line model.
    
    Parameters
    ----------
    samples : (N,4)
        Columns are amplitude, sigma, lambda, centre.
    """


    amplitude = samples[:, 0][:, None]
    sigma = samples[:, 1][:, None]
    lam = samples[:, 2][:, None]
    centre = samples[:, 3][:, None]
    wavelength = WAVELENGTH[None, :]
    # Need these [:, None] to vectorise the inputs into the shape (len(samples), 1) like a column vector 
    # so that it can be broadcast with wavelength which is now shape (1,7000)

    
    profile = gaussXExp(wavelength, amplitude,
                            sigma, lam,
                            centre)

    BL_conv = line_integral(WAVELENGTH, profile, redshift, COSMO)
    # Does not need vectorised wavelength here, hence using the global variable version which is a 1D array
    # Multiply by 130 for bolometric luminosity

    return BL_conv * 130

### for gaussian (can also use mathematical relation)
"""
def L_BH_gauss(samples, redshift):
    
    Calculates BH Halpha flux for gaussian broad line model. Needs parameters with {gauss} at the start.

    Parameters
    ----------
    samples : (N,3)
        Columns are amplitude, sigma, centre.

    Returns
    -------
    luminosity : (N,)
    

    amplitude = samples[:, 0][:, None]
    sigma = samples[:, 1][:, None]
    centre = samples[:, 2][:, None]
    wavelength = WAVELENGTH[None, :]

    profile = gaussian(wavelength, amplitude,
                            sigma,
                            centre)

    BL_gauss = line_integral(WAVELENGTH, profile, redshift, COSMO)
    # Does not need vectorised wavelength here, hence using the global variable version which is a 1D array
    
    return BL_gauss
"""
def L_BH_gauss(samples, redshift):
    """
    Calculates BH Halpha luminosity in erg/s using the analytical integral of the
    Gaussian profile.

    Parameters
    ----------
    samples : (N,3)
        Columns are amplitude, sigma, centre.

    Returns
    -------
    luminosity : (N,)
    """

    amplitude = samples[:, 0]
    sigma = samples[:, 1]
    centre = samples[:, 2]

    # Integral of Gaussian
    integrated_flux = amplitude * sigma * np.sqrt(2*np.pi)

    # Convert F_lambda -> F_nu
    integrated_flux *= 3e18 / centre**2

    # SI units
    integrated_flux *= 1e-29

    distance = COSMO.comoving_distance(redshift).value

    factor = (
        4*np.pi
        * (distance * 3.086e22 * 1e2)**2
        * (1 + redshift)
    )

    BL_gauss = integrated_flux * factor
    # L=A*σ*sqrt(2π)*​(3×10^18 / mu^2​)*10^−29×4πD^2(1+z).

    return BL_gauss * 130

In [29]:
def calculate_BL_both_models(table):
    """
    Calculates BH Halpha luminosity for both gaussian and convolved model using Shay's functions and her model data. 
    Appends it to table.
    """
    conv_amplitude16 = np.asarray(table["amplitude16"], dtype=float)
    conv_amplitude50 = np.asarray(table["amplitude50"], dtype=float)
    conv_amplitude84 = np.asarray(table["amplitude84"], dtype=float)
    conv_sigma16 = np.asarray(table["sigma16"], dtype=float)
    conv_sigma50 = np.asarray(table["sigma50"], dtype=float)
    conv_sigma84 = np.asarray(table["sigma84"], dtype=float)
    conv_centre16 = np.asarray(table["centre16"], dtype=float)
    conv_centre50 = np.asarray(table["centre50"], dtype=float)
    conv_centre84 = np.asarray(table["centre84"], dtype=float)
    conv_lambda16 = np.asarray(table["lambda16"], dtype=float)
    conv_lambda50 = np.asarray(table["lambda50"], dtype=float)
    conv_lambda84 = np.asarray(table["lambda84"], dtype=float)

    redshift = np.asarray(table["redshift_2"], dtype=float)

    gauss_amplitude16 = np.asarray(table["amplitude16"], dtype=float)
    gauss_amplitude50 = np.asarray(table["amplitude50"], dtype=float)
    gauss_amplitude84 = np.asarray(table["amplitude84"], dtype=float)
    gauss_sigma16 = np.asarray(table["sigma16"], dtype=float)
    gauss_sigma50 = np.asarray(table["sigma50"], dtype=float)
    gauss_sigma84 = np.asarray(table["sigma84"], dtype=float)
    gauss_centre16 = np.asarray(table["centre16"], dtype=float)
    gauss_centre50 = np.asarray(table["centre50"], dtype=float)
    gauss_centre84 = np.asarray(table["centre84"], dtype=float)
    

    gauss_L_BH = np.empty(len(redshift))
    gauss_L_BH_upper_err = np.empty(len(redshift))
    gauss_L_BH_lower_err = np.empty(len(redshift))

    conv_L_BH = np.empty(len(redshift))
    conv_L_BH_upper_err = np.empty(len(redshift))
    conv_L_BH_lower_err = np.empty(len(redshift))

     # I had to loop over the best_mass array because the uncertainty_propagation function only accepts single values for each parameter.

    for i in range(len(redshift)):

        gauss_params = [
            (gauss_amplitude16[i], gauss_amplitude50[i], gauss_amplitude84[i]),
            (gauss_sigma16[i], gauss_sigma50[i], gauss_sigma84[i]),
            (gauss_centre16[i], gauss_centre50[i], gauss_centre84[i])
        ]

        gauss_L_BH[i], gauss_L_BH_upper_err[i], gauss_L_BH_lower_err[i]  = uncertainty_propagation(gauss_params, 
                                                                                                   lambda samples: L_BH_gauss(samples, redshift[i])
                                                                                                   )
        # Need lambda because it simply creates a new function that "remembers" the redshift to keep the input format of L_BH_gauss function.

        conv_params = [
            (conv_amplitude16[i], conv_amplitude50[i], conv_amplitude84[i]),
            (conv_sigma16[i], conv_sigma50[i], conv_sigma84[i]),
            (conv_lambda16[i], conv_lambda50[i], conv_lambda84[i]),
            (conv_centre16[i], conv_centre50[i], conv_centre84[i])
        ]

        conv_L_BH[i], conv_L_BH_upper_err[i], conv_L_BH_lower_err[i] = uncertainty_propagation(conv_params,
                                                                                               lambda samples: L_BH_conv(samples, redshift[i])
                                                                                               )
    
    table["L_BH_gauss"] = gauss_L_BH
    table["L_BH_gauss_upper_err"] = gauss_L_BH_upper_err
    table["L_BH_gauss_lower_err"] = gauss_L_BH_lower_err
    table["L_BH_conv"] = conv_L_BH
    table["L_BH_conv_upper_err"] = conv_L_BH_upper_err
    table["L_BH_conv_lower_err"] = conv_L_BH_lower_err

    return table

In [30]:
def filter_best_bh_mass(table):
    """
    Filter the input table to select the best black hole mass based on BIC comparison.

    Parameters
    ----------
    table : astropy.table.Table
        Input table containing columns for conv_BIC, gauss_BIC, bh_mass_conv, and bh_mass_gauss.

    Returns
    -------
    filtered_table : astropy.table.Table
        Table with additional columns for best black hole mass, the model used, its fwhm, my L_Ha, Shays L_Ha, all with errors.
    """
    table = calculate_fwhm(table)
    table = calculate_BL_both_models(table)

    conv_bic = np.asarray(table["conv_BIC"], dtype=float)
    conv_mass = np.asarray(table["bh_mass_conv"], dtype=float)
    conv_mass_upper_err = np.asarray(table["bh_mass_conv_err84"], dtype=float)
    conv_mass_lower_err = np.asarray(table["bh_mass_conv_err16"], dtype=float)
    conv_fwhm = np.asarray(table["fwhm"], dtype=float)
    conv_fwhm_upper_err = np.asarray(table["fwhm_upper_err"], dtype=float)
    conv_fwhm_lower_err = np.asarray(table["fwhm_lower_err"], dtype=float)
    conv_L_BH = np.asarray(table["L_BH_conv"], dtype=float)
    conv_L_BH_upper_err = np.asarray(table["L_BH_conv_upper_err"], dtype=float)
    conv_L_BH_lower_err = np.asarray(table["L_BH_conv_lower_err"], dtype=float)                                    

    gauss_bic = np.asarray(table["gauss_BIC"], dtype=float)
    gauss_mass = np.asarray(table["bh_mass_gauss"], dtype=float)
    gauss_mass_upper_err = np.asarray(table["bh_mass_err84"], dtype=float)
    gauss_mass_lower_err = np.asarray(table["bh_mass_err16"], dtype=float)
    gauss_fwhm = np.asarray(table["gauss_fwhm"], dtype=float)
    gauss_fwhm_upper_err = np.asarray(table["gauss_fwhm_upper_err"], dtype=float)
    gauss_fwhm_lower_err = np.asarray(table["gauss_fwhm_lower_err"], dtype=float)
    gauss_L_BH = np.asarray(table["L_BH_gauss"], dtype=float)
    gauss_L_BH_upper_err = np.asarray(table["L_BH_gauss_upper_err"], dtype=float)
    gauss_L_BH_lower_err = np.asarray(table["L_BH_gauss_lower_err"], dtype=float)  

    use_conv = np.full(conv_bic.shape, False, dtype=bool)
    mask_both = ~np.isnan(conv_bic) & ~np.isnan(gauss_bic)
    use_conv[mask_both] = conv_bic[mask_both] < (gauss_bic[mask_both] - 8)
    use_conv[np.isnan(gauss_bic) & ~np.isnan(conv_bic)] = True

    best_mass = np.where(use_conv, conv_mass, gauss_mass)
    best_mass_upper_err = np.where(use_conv, conv_mass_upper_err, gauss_mass_upper_err)
    best_mass_lower_err = np.where(use_conv, conv_mass_lower_err, gauss_mass_lower_err)
    # These are in log10(Msolar) units
    best_model = np.where(use_conv, "conv", "gauss")
    best_fwhm = np.where(use_conv, conv_fwhm, gauss_fwhm)
    best_fwhm_upper_err = np.where(use_conv, conv_fwhm_upper_err, gauss_fwhm_upper_err)
    best_fwhm_lower_err = np.where(use_conv, conv_fwhm_lower_err, gauss_fwhm_lower_err)
    best_L_BH = np.where(use_conv, conv_L_BH, gauss_L_BH)
    best_L_BH_upper_err = np.where(use_conv, conv_L_BH_upper_err, gauss_L_BH_upper_err)
    best_L_BH_lower_err = np.where(use_conv, conv_L_BH_lower_err, gauss_L_BH_lower_err)

    
    table["bh_mass_best"] = best_mass
    table["bh_mass_best_upper_err"] = best_mass_upper_err
    table["bh_mass_best_lower_err"] = best_mass_lower_err
    table["bh_mass_best_model"] = best_model
    table["bh_mass_best_fwhm"] = best_fwhm
    table["bh_mass_best_fwhm_upper_err"] = best_fwhm_upper_err
    table["bh_mass_best_fwhm_lower_err"] = best_fwhm_lower_err
    table["L_BH_best"] = best_L_BH
    table["L_BH_best_upper_err"] = best_L_BH_upper_err
    table["L_BH_best_lower_err"] = best_L_BH_lower_err
    
    return table

In [31]:
Ha_table = filter_best_bh_mass(Ha_table)
# Shay's Ha line fitting table with the best black hole mass selected based on BIC comparison
NII_table = filter_best_bh_mass(NII_table)

fields = [
    "ra",
    "dec",
    "redshift_2",
    "bh_mass_best_model",
    "bh_mass_best",
    "bh_mass_best_upper_err",
    "bh_mass_best_lower_err",
    "bh_mass_best_fwhm",
    "bh_mass_best_fwhm_upper_err",
    "bh_mass_best_fwhm_lower_err",
    "L_BH_best",
    "L_BH_best_upper_err",
    "L_BH_best_lower_err",
]
# bh masses and errs are NOT! log10, they are natural!!

HA_DJA_TABLE = Ha_table[fields]
# Data I want from Shay's fitting of Halpha and NII lines

NII_DJA_TABLE = NII_table[fields]
# Data I want from Shay's fitting of Halpha and NII lines



Ha_BAGPIPES_table = open_shay_table("/nvme/scratch/work/alberttg/Summer_project/Ha_mass_BIC_BAGPIPES_DJA_err")
NII_BAGPIPES_table = open_shay_table("/nvme/scratch/work/alberttg/Summer_project/NII_mass_BIC_BAGPIPES_DJA_err")
# Contains all of Shay's photometry outputs from BAGPIPES runs of Halpha data

bagpipes_fields = ['#ID','stellar_mass_16', 'stellar_mass_50', 'stellar_mass_84']
# Will need to add more later probably

HA_BAGPIPES_TABLE = Ha_BAGPIPES_table[bagpipes_fields]
NII_BAGPIPES_TABLE = NII_BAGPIPES_table[bagpipes_fields]
# I dont know how to combine these with Shay's DJA tables because she was missing photometry for some galaxies, 
# therefore the bapgipes tables are shorter than DJA tables

In [32]:
def check_redshifts(tbl):

    if "z" in tbl.colnames and "redshift_2" in tbl.colnames:
            z_vals = np.asarray(tbl["z"], dtype=float)
            redshift_vals = np.asarray(tbl["redshift_2"], dtype=float)
            
            dz = z_vals - redshift_vals
            dz_norm = (z_vals - redshift_vals)/(1 + redshift_vals)

            print("Median dz/(1+z):", np.median(dz_norm))
            print("Scatter dz/(1+z):", np.std(dz_norm))

            t, p = ttest_rel(z_vals, redshift_vals)
            print("p value = ",p)         

            plt.figure(figsize=(7,5))
            plt.scatter(redshift_vals, dz_norm, alpha=0.7)
            plt.axhline(0, color='k', ls='--')
            plt.xlabel(r"${H\alpha} redshift$")
            plt.ylabel(r"$z_{sys} - z_{H\alpha}$")
            plt.grid(alpha=0.3)
            plt.show()
            
    return

# check_redshifts(Ha_table)

# check_redshifts(NII_table)

In [ ]:
def open_catalogue_data(path):
  """
  Reads in a catalogue of data from a FITS file and returns the data as an Astropy Table.
  """

  hdul = fits.open(path)
  hdu_names = [hdu.name for hdu in hdul]

  # sex_cat = [hdu for hdu, name in zip(hdul, hdu_names) if name == "OBJECTS"][0]
    
  # EPOCHS series (Conselice+24, Adams+24, Austin+25, Harvey+25)

  sex_tab = Table.read(path, hdu = "OBJECTS")
  sex_tab_colnames = sex_tab.colnames
  # print(sex_tab_colnames)
  # sky position (Ra = ALPHA_J2000, Dec = DELTA_J2000)
  # flux columns: FLUX_APER_{band}_aper_corr_Jy [Jy]
  # flux error columns: FLUXERR_APER_{band}_loc_depth_5pc_Jy [Jy]

  
  eazy_tab = Table.read(path, hdu = "EAZY_SFHZ_BLUE_AGN")
  eazy_tab_colnames = eazy_tab.colnames

  # ID = "IDENT" same as "NUMBER" from sex
  # redshift, z = 'zbest_sfhz_blue_agn_zfree' not median!
  # redshift_errors = ['zbest_16_sfhz_blue_agn_zfree', 'zbest_84_sfhz_blue_agn_zfree']
  # redshift - redshift errors[0] = lower sigma
  # redshift - redshift errors[1] = upper sigma


  eazy_properties_tab = Table.read(path, hdu = 'PROPERTIES_EAZY_SFHZ_BLUE_AGN')
  eazy_properties_tab_colnames = eazy_properties_tab.colnames

  # beta/Muv

  selection_tab = Table.read(path, hdu = "SELECTION")
  selection_tab_colnames = selection_tab.colnames

  # EPOCHS_good_EAZY_sfhz_blue_agn_0.32as == True

  bagpipes_tab = Table.read(path, hdu = 'BAGPIPES_SFH_CONT_BURSTY_ZEAZYSFHZBLUEAGN_3.0,10.0MYR_CALZETTI_LOG_10_Z_LOG_10_BPASS_ZGAUSS_3.0SIG')
  bagpipes_tab_colnames = bagpipes_tab.colnames

  # only run on EPOCHS_good_EAZY_sfhz_blue_agn_0.32as == True AND z > 6.5
  # galaxy properties
  
  return sex_tab, eazy_tab, eazy_properties_tab, selection_tab, bagpipes_tab

In [34]:
def match_galaxies_by_coord(
    sex_tab, eazy_tab, DJA_table, shay_bagpipes_table,
    filters
):
    """
    Match a list of RA/Dec coordinates to a catalogue and extract
    selected information.

    Parameters
    ----------
    sex_tab : astropy.table.Table
        Catalogue table.

    DJA_table : astropy.table.Table
        My shortened version of Shay's line fitting table of galaxies to match.

    filters : list of str
        List of filter names used in the flux column names.

    match_radius : astropy.units.Quantity
        Maximum separation for a successful match.

    Returns
    -------
    astropy.table.Table
        Table containing matched objects and requested columns.
    """

    galaxy_ra = DJA_table["ra"]
    galaxy_dec = DJA_table["dec"]

    match_radius = 0.32 * u.arcsec  # Set the match radius to 0.32 arcseconds
    # Galaxy coordinates
    gal_coords = SkyCoord(
        ra=np.asarray(galaxy_ra) * u.deg,
        dec=np.asarray(galaxy_dec) * u.deg
    )

    # Catalogue coordinates
    sex_tab_coords = SkyCoord(
        ra=sex_tab["ALPHA_J2000"] * u.deg,
        dec=sex_tab["DELTA_J2000"] * u.deg
    )
    # Set to deg

    # Find nearest catalogue object for each galaxy
    idx, sep2d, sep3d = gal_coords.match_to_catalog_sky(sex_tab_coords)
    # indx is the index of the nearest catalogue that matches the galaxy
    # sep2d is the on-sky separation between the galaxy and the matched catalogue object
    # sep3d is the 3D separation, which is not used here

    # Keep only matches within match_radius, I will set to 0.32 as
    good = sep2d < match_radius
    bad = sep2d > match_radius

    # Need to check that I have all the different flux filters that I want to extract from the catalogue
    flux_col = {F: f"FLUX_APER_{F}_aper_corr_Jy" for F in filters}
    fluxerr_col = {F: f"FLUXERR_APER_{F}_loc_depth_5pc_Jy" for F in filters}

    flux_auto = {F: f"FLUX_AUTO_{F}" for F in filters}
    flux_radius = {F: f"FLUX_RADIUS_{F}" for F in filters}

    a_image = {F: f"A_IMAGE_{F}" for F in filters}
    b_image = {F: f"B_IMAGE_{F}" for F in filters}
    theta_image = {F: f"THETA_IMAGE_{F}" for F in filters}


    matched_rows = []
    unmatched_rows = []

    for i in np.where(good)[0]:

        matched_rows.append({
            "SURVEY_ID": sex_tab["SURVEY_ID"][idx[i]],
            "SURVEY": sex_tab["SURVEY"][idx[i]],
            "ALPHA_J2000": sex_tab["ALPHA_J2000"][idx[i]],
            "DELTA_J2000": sex_tab["DELTA_J2000"][idx[i]],
            "REDSHIFT": DJA_table["redshift_2"][i],
            **{f"FLUX_{F}": sex_tab[flux_col[F]][idx[i]] for F in filters},
            **{f"FLUXERR_{F}": sex_tab[fluxerr_col[F]][idx[i]] for F in filters},
            **{f"FLUX_AUTO_{F}": sex_tab[flux_auto[F]][idx[i]] for F in filters},
            **{f"FLUX_RADIUS_{F}": sex_tab[flux_radius[F]][idx[i]] for F in filters},
            **{f"A_IMAGE_{F}": sex_tab[a_image[F]][idx[i]] for F in filters},
            **{f"B_IMAGE_{F}": sex_tab[b_image[F]][idx[i]] for F in filters},
            **{f"THETA_IMAGE_{F}": sex_tab[theta_image[F]][idx[i]] for F in filters},
            "log10_BH_MASS": DJA_table["bh_mass_best"][i],
            "log10_BH_MASS_UPPER_ERR": DJA_table["bh_mass_best_upper_err"][i],
            "log10_BH_MASS_LOWER_ERR": DJA_table["bh_mass_best_lower_err"][i],
            "FWHM": DJA_table["bh_mass_best_fwhm"][i],
            "FWHM_UPPER_ERR": DJA_table["bh_mass_best_fwhm_upper_err"][i],
            "FWHM_LOWER_ERR": DJA_table["bh_mass_best_fwhm_lower_err"][i],
            "BEST_MODEL": DJA_table["bh_mass_best_model"][i],
            "L_BH_BOL": DJA_table["L_BH_best"][i],
            "L_BH_BOL_UPPER_ERR": DJA_table["L_BH_best_upper_err"][i],
            "L_BH_BOL_LOWER_ERR": DJA_table["L_BH_best_lower_err"][i]
        })
        # Might need to add more columns here, but for now I will just extract the fluxes and flux errors for the filters I want

    matched_table = Table(rows=matched_rows)

    for i in np.where(bad)[0]:

        unmatched_rows.append({
            "RA": DJA_table["ra"][i],
            "DEC": DJA_table["dec"][i],
            "REDSHIFT": DJA_table["redshift_2"][i]
        })

    unmatched_table = Table(rows=unmatched_rows)

    # Remove duplicate rows with the same SEXTRACTOR number, keeping the first match only.
    _, unique_indices = np.unique(matched_table["SURVEY_ID"], return_index=True)
    matched_table = matched_table[np.sort(unique_indices)]

    # print("matched table:")
    # print(matched_table)
    return matched_table, unmatched_table

In [35]:
def check_table_lengths(DJA_table, shay_bagpipes_table, paths_to_files, filters):
  """
  Check the lengths of the original table from Shay against the new table made from fiding matches in the filenames.
  If lengths the same, all galaxies found and I want the outputted new table. If lengths different, 
  I want to check the next file in the list of paths_to_files. If new table has too many galaxies, then 
  duplicates may have been found and I want the outputted new table (I think).

  Parameters:
  original_table (astropy.table.Table): The original table to compare against, from Shay's data.
  paths_to_files (list of str): List of file paths to read and check.

  Returns:
  new_table (astropy.table.Table): The matched table from the first file that has the
  same length as the original table, or the first file that has more galaxies than the original table,
  or the last checked matched table if no file contains all galaxies.
  """
  

  for path in paths_to_files:
    sex_tab, eazy_tab, eazy_properties_tab, selection_tab, bagpipes_tab  = open_catalogue_data(path)

    new_table, unmatched_table = match_galaxies_by_coord(sex_tab, eazy_tab, DJA_table, shay_bagpipes_table, filters)

    print(f"Checking catalogue: {path}")
    print(f"Length of Shay's table: {len(DJA_table)}")
    print(f"Length of matched new_table: {len(new_table)}")

    if len(new_table) == len(DJA_table):
        done = True
        print("All galaxies found. No need to check further files.")
        return new_table
        break
    elif len(new_table) < len(DJA_table):
        print("Not all galaxies found in this file. Trying the next file...")
        continue
    if path == paths_to_files[-1]:
        print("Reached the last file in the list. Returning last matched table, even though it has fewer galaxies than Shay's table.")
        return new_table
    else:
        print("Duplicate galaxies may have been found.")
        return new_table

  return new_table, unmatched_table
  

In [36]:
if __name__ == "__main__":

    """
    general structure of the catalogues is 
    /raid/scratch/work/austind/GALFIND_WORK/Catalogues/{version}/ACS_WFC+NIRCam/{survey}/
    (0.32)as/{survey}_MASTER_Sel-F277W+F356W+F444W_{version}.fits
    """

    """
    list_of_paths = [
        "/raid/scratch/work/austind/GALFIND_WORK/Catalogues/v12_psfmatch_F444W_empirical+v13_psfmatch_F444W_empirical+v14_psfmatch_F444W_empirical/ACS_WFC+NIRCam/EPOCHS-v2_EPOCHS_good_EAZY_sfhz_blue_agn_zfree/(0.32)as/EPOCHS-v2_EPOCHS_good_EAZY_sfhz_blue_agn_zfree_MASTER_Sel-F277W+F356W+F444W_v12_psfmatch_F444W_empirical+v13_psfmatch_F444W_empirical+v14_psfmatch_F444W_empirical.fits",
        "/raid/scratch/work/austind/GALFIND_WORK/Catalogues/v12_psfmatch_F444W_empirical+v13_psfmatch_F444W_empirical+v14_psfmatch_F444W_empirical/ACS_WFC+NIRCam/EPOCHS-v2/(0.32)as/EPOCHS-v2_MASTER_Sel-F277W+F356W+F444W_v12_psfmatch_F444W_empirical+v13_psfmatch_F444W_empirical+v14_psfmatch_F444W_empirical.fits"]
        # Add more file paths as needed
    # The first is the EPOCHS file where galaxies have been filtered by robust galaxy selection criteria
    # The second is the EPOCHS file where galaxies have not been filtered, so is much larger
    I wont use these paths because they are on the raider server and still being tweaked by Duncan, 
    so I will use the paths on my nvme server instead.
    """


    list_of_paths = ["/nvme/scratch/work/alberttg/Summer_project/EPOCHS-v2_MASTER_Sel-F277W+F356W+F444W_v12_psfmatch_F444W_empirical+v13_psfmatch_F444W_empirical+v14_psfmatch_F444W_empirical.fits"]
    # This is ONLY the EPOCHS file where galaxies have not been filtered, so is much larger

    FILTERS =["F115W", "F150W", "F200W", "F277W", "F356W", "F444W", "F606W", "F814W"]

    Ha_table_broad_line_data, Ha_unmatched = check_table_lengths(HA_DJA_TABLE, HA_BAGPIPES_TABLE, list_of_paths, FILTERS)
    # Although I have included the bagpipes table in this function I havent actually added any data from it because some galaxies were missing
    # photometry when shay ran bagpipes
    # nvm

    NII_table_broad_line_data, NII_unmatched = check_table_lengths(NII_DJA_TABLE, NII_BAGPIPES_TABLE, list_of_paths, FILTERS)

    combined_table = vstack(
    [Ha_table_broad_line_data, NII_table_broad_line_data],
    join_type="exact"
    )
    combined_table.write("Ha_and_NII_broad_line_data.fits", format="fits", overwrite=True)
    # My table of Shay's redshift, bh_mass and error, catalogues photometry and survey IDs

    SURVEY_ID_table = combined_table["SURVEY_ID", "SURVEY"]
    SURVEY_ID_table.write("Ha_and_NII_broad_line_SURVEY_IDs.csv", format="ascii.csv", overwrite=True)
    # For making cutouts

    unmatched_combined = vstack([Ha_unmatched, NII_unmatched], join_type="exact")
    unmatched_combined.write("unmatched_galaxies.csv", format="ascii.csv", overwrite=True)



    


['SURVEY_ID', 'X_IMAGE', 'Y_IMAGE', 'ALPHA_J2000', 'DELTA_J2000', 'MAG_APER_F277W+F356W+F444W', 'FLUX_APER_F277W+F356W+F444W', 'MAGERR_APER_F277W+F356W+F444W', 'FLUXERR_APER_F277W+F356W+F444W', 'MAG_AUTO_F277W+F356W+F444W', 'MAGERR_AUTO_F277W+F356W+F444W', 'FLUX_AUTO_F277W+F356W+F444W', 'FLUXERR_AUTO_F277W+F356W+F444W', 'MAG_BEST_F277W+F356W+F444W', 'MAGERR_BEST_F277W+F356W+F444W', 'MAG_ISO_F277W+F356W+F444W', 'MAGERR_ISO_F277W+F356W+F444W', 'KRON_RADIUS_F277W+F356W+F444W', 'FLUX_RADIUS_F277W+F356W+F444W', 'FWHM_IMAGE_F277W+F356W+F444W', 'CLASS_STAR_F277W+F356W+F444W', 'SNR_WIN_F277W+F356W+F444W', 'ELONGATION_F277W+F356W+F444W', 'THETA_IMAGE_F277W+F356W+F444W', 'A_IMAGE_F277W+F356W+F444W', 'B_IMAGE_F277W+F356W+F444W', 'FLAGS_F277W+F356W+F444W', 'ISOAREA_IMAGE_F277W+F356W+F444W', 'MAG_APER_F435W', 'FLUX_APER_F435W', 'MAGERR_APER_F435W', 'FLUXERR_APER_F435W', 'MAG_AUTO_F435W', 'MAGERR_AUTO_F435W', 'FLUX_AUTO_F435W', 'FLUXERR_AUTO_F435W', 'MAG_BEST_F435W', 'MAGERR_BEST_F435W', 'MAG_ISO_F4

Checking catalogue: /nvme/scratch/work/alberttg/Summer_project/EPOCHS-v2_MASTER_Sel-F277W+F356W+F444W_v12_psfmatch_F444W_empirical+v13_psfmatch_F444W_empirical+v14_psfmatch_F444W_empirical.fits
Length of Shay's table: 160
Length of matched new_table: 106
Not all galaxies found in this file. Trying the next file...
